# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring** (with a supervised model underneath, but evaluated as a ranking).

My lane's question is "which visible pages should a reviewer look at first?" In the framing map,
"which ones first?" is a ranking/scoring task: I score every visible page by how far it under
captures clicks for its search position, then order the pages so the review team works top down
against its limited capacity. It is not primarily classification, because a hard yes/no on 30,000
pages is less useful than an ordered shortlist; it is not clustering, because I am not looking for
"types" of pages; and it is not pure signal analysis, because the deliverable is an actionable
ordered queue, not just a correlation report.

Underneath the ranking I can train a supervised model (logistic regression or a tree) to produce
the score, but I will judge it the way the queue is actually used — by the quality of its top of
list — not by raw accuracy.

In [1]:
# Setup + load. Output of this lane is a RANKING, so I size the population that gets ranked.
import os, sys, subprocess
import pandas as pd, numpy as np

if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The ranked population: visible pages with enough exposure to matter.
vis = df[(df["impressions_90d"] >= 100) & (df["ctr"].notna())].copy()
print("Task type: ranking / scoring  ('which pages first?')")
print(f"Pages to rank (visible, impressions_90d >= 100): {len(vis):,}")
print("Output: an ordered review queue, judged by the quality of its top of list.")

Task type: ranking / scoring  ('which pages first?')
Pages to rank (visible, impressions_90d >= 100): 22,006
Output: an ordered review queue, judged by the quality of its top of list.


## 2. Target or proxy

**Starter proxy (what I can build today): a position adjusted CTR gap.** For each visible page I
compare its CTR to the median CTR of its own position tier, and treat a page as an
opportunity when it earns less than half its tier's median CTR. This is a *defined* proxy, not an
observed outcome, and I am naming that weakness on purpose: because the rule defines the label, a
model trained on it alone would partly be learning my rule. So in the starter slice I use the gap as
a transparent scoring target and a sanity check, not as ground truth.

**Stronger target for the capstone (observed, future looking): a later CTR outcome.** Using the
warehouse daily facts I can define a real outcome measured *after* a decision point, for example:
take a page's features from a prior 90 day window, then label whether its CTR stays below its
position tier's expectation (or fails to improve) over the next 30 days. That label is an observed
future outcome, which is exactly what the framing rule "the target must be observed, not defined"
asks for. The starter proxy is the training-wheels version; the observed future label is the real
one I will build once I have the warehouse.

In [2]:
# Build the starter PROXY target and show it is honestly balanced (not all-or-nothing).
vis["tier_median_ctr"] = vis.groupby("position_tier")["ctr"].transform("median")
vis["ctr_gap"] = vis["ctr"] - vis["tier_median_ctr"]                 # scoring signal (continuous)
vis["under_captures"] = (vis["ctr"] < 0.5 * vis["tier_median_ctr"]).astype(int)  # proxy label (0/1)

bal = vis["under_captures"].value_counts(normalize=True).round(3)
print("Proxy label 'under_captures' (defined from a position-adjusted rule):")
print(f"  positive (under-capturing): {bal.get(1,0)*100:.1f}%")
print(f"  negative (normal for tier): {bal.get(0,0)*100:.1f}%")
print("\nNote: this is a DEFINED proxy from current-window data. The capstone target is an")
print("OBSERVED future outcome (next-window CTR), which the warehouse daily facts make possible.")

Proxy label 'under_captures' (defined from a position-adjusted rule):
  positive (under-capturing): 32.9%
  negative (normal for tier): 67.1%

Note: this is a DEFINED proxy from current-window data. The capstone target is an
OBSERVED future outcome (next-window CTR), which the warehouse daily facts make possible.


## 3. Success metric

**Primary metric: Precision@K** (Precision@20 and Precision@50), plus a **by hand review of the top
20**. The queue is used by a reviewer who can only open so many pages, so the only thing that matters
is how many of the *top K* pages are genuine CTR opportunities that a human confirms — not overall
accuracy across all 30,000 pages. Precision@K matches the decision exactly.

**What "good" means.** Two bars. First, the ranked score must beat a naive baseline that just sorts
by lowest raw CTR (the fixed rule), on Precision@K. Second, in the by hand top 20 review, most flagged
pages must look like real opportunities (strong position, real volume, weak CTR) rather than obvious
noise, brand queries, or intent mismatches. I define these bars *before* training, so "good" cannot
be redrawn after the fact.

In [3]:
# Prove the metric is computable TODAY: score a naive baseline and read its Precision@K.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))     # highest score first
    return np.asarray(labels)[order[:k]].mean()

y = vis["under_captures"].values
# Naive fixed-rule baseline: "most suspicious = lowest raw CTR" (ignores position).
naive_score = -vis["ctr"].values
for k in (20, 50):
    p = precision_at_k(naive_score, y, k)
    print(f"Naive 'lowest raw CTR' baseline  Precision@{k}: {p:.3f}   (~{round(p*k)} of top {k})")
print("\nThe metric is computable today. But notice the score is deceptively high: because the")
print("PROXY label is itself defined from CTR, sorting by CTR trivially 'wins' against it. That is")
print("exactly why the real success bar must be Precision@K against an OBSERVED future outcome, plus")
print("a by-hand top-20 review -- not agreement with a rule I wrote myself.")

Naive 'lowest raw CTR' baseline  Precision@20: 1.000   (~20 of top 20)
Naive 'lowest raw CTR' baseline  Precision@50: 0.860   (~43 of top 50)

The metric is computable today. But notice the score is deceptively high: because the
PROXY label is itself defined from CTR, sorting by CTR trivially 'wins' against it. That is
exactly why the real success bar must be Precision@K against an OBSERVED future outcome, plus
a by-hand top-20 review -- not agreement with a rule I wrote myself.


## 4. The unit of analysis, as a real dataframe

**One row = one visible content page.** Below is the actual slice: each row is a single page
(`content_id`) belonging to a pseudonymized client, with the observable signals the score uses and
the sketched target columns (`ctr_gap` as the continuous score, `under_captures` as the 0/1 proxy).
This is the grain everything else is built on.

In [4]:
# Show the grain as a real dataframe: one row per visible page, with the sketched target columns.
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr",
        "avg_position", "position_tier", "tier_median_ctr", "ctr_gap", "under_captures"]
preview = vis.sort_values("ctr_gap").loc[:, cols].head(8)   # most under-capturing first
print(f"Grain: one row = one visible content page. Rows in slice: {len(vis):,}\n")
print(preview.to_string(index=False))

Grain: one row = one visible content page. Rows in slice: 22,006

          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position position_tier  tier_median_ctr  ctr_gap  under_captures
content_2a228ce7aa1b client_8527a891e2              155           0  0.0           7.5        page_1             0.23    -0.23               1
content_5ef76f383560 client_f369cb89fc              973           0  0.0           3.2        page_1             0.23    -0.23               1
content_12018e6ac413 client_8722616204              144           0  0.0           8.3        page_1             0.23    -0.23               1
content_3bc49e1805db client_bbb965ab0c              683           0  0.0           6.8        page_1             0.23    -0.23               1
content_b0bbe3291c4c client_f369cb89fc              106           0  0.0           7.3        page_1             0.23    -0.23               1
content_9f8ff0fcf40b client_19581e27de              262           0  0.0    

## 5. Why ML beats a fixed rule here

A single CTR threshold cannot express the problem, because "good CTR" is completely different at each
position. The numbers below show it: a flat `ctr < 0.5` rule flags about 18,600 pages, most of them
simply because they rank low, while the position adjusted view isolates about 7,200 real gaps. The
flat rule floods the queue with roughly 11,000 pages that are *normal for their position*, and among
strong position pages it fires almost indiscriminately (about 5,400 flags) instead of finding the
2,200 that genuinely under convert.

To rank fairly you need a different expected CTR per position tier, then you need to weigh several
signals together — position, volume, intent, content type, engagement — and filter low volume noise.
That is many tangled signals with tier specific baselines, which is exactly the case where a learned,
position aware score earns its place over an if statement. The rule is not just less accurate; it is
answering the wrong question.

In [5]:
# A fixed threshold vs a position-adjusted view: how badly does one CTR cutoff misfire?
flat = (vis["ctr"] < 0.5)
gap  = (vis["under_captures"] == 1)
print("Flat rule  (ctr < 0.5) flags        :", int(flat.sum()))
print("Position-adjusted gap flags         :", int(gap.sum()))
print("  flat flags that are NOT a real gap:", int((flat & ~gap).sum()), "(normal-for-position, wrongly flagged)")

strong = vis[vis["position_tier"].isin(["top_3", "striking"])]
print("\nAmong strong-position pages (top_3 / striking):", len(strong))
print("  flat rule ctr<0.5 flags           :", int((strong["ctr"] < 0.5).sum()), "(fires almost indiscriminately)")
print("  position-adjusted under_captures  :", int((strong["under_captures"] == 1).sum()), "(the real opportunities)")
print("\nObserved / directional: one global CTR threshold cannot separate 'normal for this position'")
print("from 'genuinely under-capturing'. Per-tier baselines plus multi-signal ranking is the job.")

Flat rule  (ctr < 0.5) flags        : 18614
Position-adjusted gap flags         : 7239
  flat flags that are NOT a real gap: 11375 (normal-for-position, wrongly flagged)

Among strong-position pages (top_3 / striking): 6436
  flat rule ctr<0.5 flags           : 5389 (fires almost indiscriminately)
  position-adjusted under_captures  : 2229 (the real opportunities)

Observed / directional: one global CTR threshold cannot separate 'normal for this position'
from 'genuinely under-capturing'. Per-tier baselines plus multi-signal ranking is the job.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.